In [1]:
import os


In [21]:
%pwd

'C:\\Users\\Gauri\\Downloads\\Abhishek\\Doc-Summarization\\DocSummarizer'

In [19]:
os.chdir("C:\\Users\\Gauri\\Downloads\\Abhishek\\Doc-Summarization\\DocSummarizer")

In [22]:

from dataclasses import dataclass
@dataclass(frozen=True)
class DataValidationConfig:
    raw_data_dir: Path
    STATUS_FILE: Path
    ALL_REQUIRED_FILES: list[str]


In [23]:


from DocSummarizer.constants import *
from DocSummarizer.utils.common import read_yaml

import os
import logging
from dataclasses import dataclass
from pathlib import Path

def create_directories_cust(dirs: list[str]):
    for dir_path in dirs:
        os.makedirs(dir_path, exist_ok=True)
        logging.info(f"Created directory: {dir_path}")


class ConfigurationManager:
    def __init__(self, config_file_path = CONFIG_FILE_PATH, params_file_path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)

        # Ensure artifacts_root is a string
        artifacts_root = (
            self.config["artifacts_root"]
            if isinstance(self.config, dict)
            else self.config.artifacts_root
        )
        # create_directories(list([str(artifacts_root)]))
        print("artifacts_root type:", type(artifacts_root))
        print("artifacts_root value:", artifacts_root)
        create_directories_cust([artifacts_root])

    
    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation

        create_directories_cust([str(config.raw_data_dir)])
        
        data_validation_config = DataValidationConfig(
            raw_data_dir = Path(config.raw_data_dir),
            STATUS_FILE = Path(config.STATUS_FILE),
            ALL_REQUIRED_FILES = config.ALL_REQUIRED_FILES
        )
        return data_validation_config


In [24]:
import os
from DocSummarizer.logging import logger

In [30]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config


    def validate_all_files(self) -> bool:
        try:
            all_files_present = True
            all_files=os.listdir(os.path.join("artifacts","data_ingestion"))
            for file_name in all_files:
                if file_name not in self.config.ALL_REQUIRED_FILES:
                    all_files_present=False
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"Required file {file_name} is missing in {all_files}")
                else:
                    all_files_present = True
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"All required files are present in {all_files}")
            return all_files_present
        except Exception as e:
            logger.exception(f"Error during file validation: {e}")
            raise e

In [31]:
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_files()
except Exception as e:
    # logger.exception(e)
    raise e

[2026-03-29 19:57:14,495: INFO: common: YAML file 'config\config.yaml' read successfully.]
[2026-03-29 19:57:14,500: INFO: common: YAML file 'params.yaml' read successfully.]
artifacts_root type: <class 'str'>
artifacts_root value: artifacts
[2026-03-29 19:57:14,507: INFO: 3850906572: Created directory: artifacts]
[2026-03-29 19:57:14,512: INFO: 3850906572: Created directory: artifacts/data_validation]


In [32]:
import DocSummarizer.components.data_validation as dv
print(dir(dv))

['DataValidation', 'DataValidationConfig', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'logger', 'os']


In [33]:
import DocSummarizer.components.data_validation as dv
print(dv.DataValidation)

<class 'DocSummarizer.components.data_validation.DataValidation'>


In [34]:
import DocSummarizer.pipeline.stage_02_data_validation as p
print(dir(p))

['ConfigurationManager', 'DataValidation', 'DataValidationPipeline', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'logger']
